In [0]:
%pip install langchain
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from databricks.vector_search.client import VectorSearchClient

load_dotenv()


True

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    results = VectorSearchClient().get_index().similarity_search(
        query_text=...,
        columns=...,
        num_results=...
    )
    docs = ...
    raw_docs = [doc[1] for doc in docs if len(doc) > 1]
    return "\n\n---\n\n".join(raw_docs)


@tool
def vector_search_tool(query: str) -> str:
    # TO-DO: Write a docstring for this function
    # TO-DO: Implement the function
    pass

@tool
def read_fallback_document(category: str) -> str:
    # TO-DO: Write a docstring for this function
    # TO-DO: Complete the function
    file_map = {
        "databricks": "path/to/databricks_info.txt",
        "azure": "path/to/azure_info.txt"
    }
    
    category = ...
    file_path = ...
    
    if not file_path:
        return f"No documentation found for category: {category}"
        
    try:
        with open(file_path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error reading document: {e}"
    

def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
        api_version=os.environ.get("OPENAI_API_VERSION"), 
        temperature=0.0
    )
    
    tools = ...
    memory = ...
    
    mcp_agent = create_react_agent(...)
    return mcp_agent

agent = build_stateful_mcp_agent()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-2592799060090138>, line 12
      8     raw_docs = [doc[1] for doc in docs if len(doc) > 1]
      9     return "\n\n---\n\n".join(raw_docs)
---> 12 @tool
     13 def vector_search_tool(query: str) -> str:
     14     # TO-DO: Write a docstring for this function
     15     # TO-DO: Implement the function
     16     pass
     18 @tool
     19 def read_fallback_document(category: str) -> str:
     20     # TO-DO: Write a docstring for this function
     21     # TO-DO: Complete the function

NameError: name 'tool' is not defined

In [0]:
# Summary: Executes queries against the agent while maintaining a consistent session thread.
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = ...
    
    for query in queries:
        print(f"\nUser: {query}")
        payload = ...
        
        response = ...
        final_message = ...
        
        print(f"Agent: {final_message}")

In [0]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-2592799060090140>, line 12
      1 validation_queries = [
      2     "What tools do you have?",
      3     "What is searchAgent-X?",
   (...)
      8     "What core features differentiate both products?"
      9 ]
     11 print("--- TESTING STATEFUL MCP AGENT ---")
---> 12 execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

NameError: name 'agent' is not defined

In [ ]:
# Bonus tasks: tools + stateful MCP agent
import os, json
from typing import Optional
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_openai import AzureChatOpenAI
from databricks.vector_search.client import VectorSearchClient

# Config (reuse main assignment names)
CATALOG = "25280091_pa3"
VECTOR_ENDPOINT = "25280091_vector_endpoint"
INDEX_NAME = f"{CATALOG}.vector_index.fixed_vector_index"
FALLBACK_DIR = f"/dbfs/Volumes/{CATALOG}/default/text_documents"

vsc = VectorSearchClient(disable_notice=True)
idx = vsc.get_index(endpoint_name=VECTOR_ENDPOINT, index_name=INDEX_NAME)


def retrieve_formatted_context(question: str, topk: int = 3) -> str:
    """Query vector index and return top-k chunk_text joined with separators."""
    res = idx.similarity_search(query_text=question, columns=["chunk_text"], num_results=topk)
    hits = res.get("result", {}).get("data_array", []) if isinstance(res, dict) else []
    raw_docs = [h[0] for h in hits if h and len(h) > 0 and h[0]]
    return "\n\n---\n\n".join(raw_docs)


@tool("vector_search_tool")
def vector_search_tool(query: str, top_k: int = 3) -> str:
    """Retrieve top-k chunks for a query from the fixed vector index."""
    ctx = retrieve_formatted_context(query, top_k)
    return ctx or "No results found"


@tool("read_fallback_document")
def read_fallback_document(category: str) -> str:
    """Read a fallback .txt document from the text_documents volume by category (e.g., databricks, azure)."""
    category = (category or "").strip().lower()
    file_map = {
        "databricks": os.path.join(FALLBACK_DIR, "databricks.txt"),
        "azure": os.path.join(FALLBACK_DIR, "azure.txt"),
    }
    file_path = file_map.get(category)
    if not file_path:
        return f"No documentation found for category: {category}"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        return f"Error reading document: {e}"


def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"),
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION") or os.environ.get("OPENAI_API_VERSION"),
        api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
        temperature=0,
    )
    tools = [vector_search_tool, read_fallback_document]
    memory = MemorySaver()
    return create_react_agent(llm, tools=tools, checkpointer=memory)

agent = build_stateful_mcp_agent()


In [ ]:
# Execute MCP queries with thread-level state (MemorySaver)
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    for query in queries:
        print(f"\nUser: {query}")
        payload = {"messages": [{"role": "user", "content": query}]}
        response = mcp_agent.invoke(payload, config=config)
        msgs = response.get("messages", []) if isinstance(response, dict) else []
        final_message = ""
        if msgs:
            content = msgs[-1].get("content", "")
            if isinstance(content, list):
                # handle OpenAI-style message parts
                final_message = "".join(x.get("text", "") if isinstance(x, dict) else str(x) for x in content)
            else:
                final_message = str(content)
        print(f"Agent: {final_message}")


validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?",
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")
